# OMOP Procedure Occurrence Table

Transforms FHIR Procedure resources into OMOP CDM `procedure_occurrence` table.

## Mapping: FHIR Procedure → OMOP Procedure_Occurrence

| OMOP Field | FHIR Source | Transformation |
|------------|-------------|----------------|
| procedure_occurrence_id | Procedure.id | Hash to integer |
| person_id | Procedure.subject | Reference to person |
| procedure_concept_id | Procedure.code | Map SNOMED/CPT to OMOP |
| procedure_date | Procedure.performedDateTime | Extract date |
| procedure_datetime | Procedure.performedDateTime | Full timestamp |
| procedure_end_date | Procedure.performedPeriod.end | End date |
| procedure_end_datetime | Procedure.performedPeriod.end | End datetime |
| procedure_type_concept_id | - | 32817 (EHR) |
| modifier_concept_id | Procedure.bodySite | Modifier concept |
| visit_occurrence_id | Procedure.encounter | Reference to visit |
| procedure_source_value | Procedure.code.coding[0].code | Original code |

## Vocabulary Mapping

| Source System | Target | Notes |
|--------------|--------|-------|
| SNOMED CT | Standard | Direct mapping |
| CPT-4 | Non-standard → SNOMED | Use concept_relationship |
| HCPCS | Non-standard → SNOMED | Use concept_relationship |
| ICD-10-PCS | Non-standard → SNOMED | Use concept_relationship |

_Note: Attach to a Serverless SQL Warehouse for execution._

In [ ]:
-- ============================================================================
-- CONFIGURATION
-- ============================================================================
DECLARE OR REPLACE VARIABLE catalog_use STRING DEFAULT 'redox_fhir';
DECLARE OR REPLACE VARIABLE silver_schema STRING DEFAULT 'bronze';
DECLARE OR REPLACE VARIABLE gold_schema STRING DEFAULT 'omop';

SET VARIABLE catalog_use = COALESCE(:catalog_use, catalog_use);
SET VARIABLE silver_schema = COALESCE(:silver_schema, silver_schema);
SET VARIABLE gold_schema = COALESCE(:gold_schema, gold_schema);

USE IDENTIFIER(catalog_use || '.' || gold_schema);
SELECT current_catalog(), current_schema();

## Create Procedure Occurrence Streaming Table

In [ ]:
DECLARE OR REPLACE VARIABLE create_procedure_stmt STRING;

SET VARIABLE create_procedure_stmt = "
CREATE OR REFRESH STREAMING TABLE procedure_occurrence (
  -- Primary key
  procedure_occurrence_id BIGINT NOT NULL COMMENT 'Unique procedure occurrence identifier'
  
  -- Person reference
  ,person_id BIGINT NOT NULL COMMENT 'Reference to person table'
  
  -- Procedure coding
  ,procedure_concept_id INT NOT NULL COMMENT 'OMOP standard concept for procedure'
  
  -- Dates
  ,procedure_date DATE NOT NULL COMMENT 'Procedure date'
  ,procedure_datetime TIMESTAMP COMMENT 'Procedure datetime'
  ,procedure_end_date DATE COMMENT 'Procedure end date'
  ,procedure_end_datetime TIMESTAMP COMMENT 'Procedure end datetime'
  
  -- Type
  ,procedure_type_concept_id INT NOT NULL DEFAULT 32817 COMMENT 'Type: 32817=EHR'
  
  -- Modifier
  ,modifier_concept_id INT DEFAULT 0 COMMENT 'Modifier concept (body site, laterality)'
  
  -- Quantity
  ,quantity INT COMMENT 'Number of procedures performed'
  
  -- References
  ,provider_id BIGINT COMMENT 'Reference to provider table'
  ,visit_occurrence_id BIGINT COMMENT 'Reference to visit_occurrence table'
  ,visit_detail_id BIGINT COMMENT 'Reference to visit_detail table'
  
  -- Source values
  ,procedure_source_value STRING COMMENT 'Original procedure code'
  ,procedure_source_concept_id INT DEFAULT 0 COMMENT 'Source vocabulary concept'
  ,modifier_source_value STRING COMMENT 'Original modifier value'
  
  -- Code system info
  ,procedure_code_system STRING COMMENT 'Source code system (SNOMED, CPT, etc.)'
  ,procedure_display STRING COMMENT 'Display text for procedure'
  
  -- Lineage
  ,fhir_procedure_uuid STRING COMMENT 'Original FHIR Procedure UUID'
  ,bundle_uuid STRING COMMENT 'Source bundle reference'
)
COMMENT 'OMOP CDM Procedure Occurrence table - Procedures from FHIR Procedure resources'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
  ,'delta.enableDeletionVectors' = 'true'
  ,'delta.enableRowTracking' = 'true'
  ,'quality' = 'gold'
  ,'pipelines.channel' = 'PREVIEW'
  ,'delta.feature.variantType-preview' = 'supported'
)
AS 
SELECT
  -- Generate integer procedure_occurrence_id
  ABS(HASH(COALESCE(id::STRING, procedure_uuid))) AS procedure_occurrence_id
  
  -- Person reference
  ,ABS(HASH(
    COALESCE(
      REGEXP_EXTRACT(subject:reference::STRING, 'Patient/(.+)', 1),
      subject:reference::STRING
    )
  )) AS person_id
  
  -- Procedure concept - placeholder using hash of code
  -- In production, join to OMOP vocabulary tables for proper concept_id
  ,COALESCE(
    ABS(HASH(code:coding[0]:code::STRING)) % 2000000000,
    0
  ) AS procedure_concept_id
  
  -- Dates
  ,COALESCE(
    CAST(TRY_CAST(performedDateTime::STRING AS TIMESTAMP) AS DATE),
    CAST(TRY_CAST(performedPeriod:start::STRING AS TIMESTAMP) AS DATE),
    CURRENT_DATE()
  ) AS procedure_date
  ,COALESCE(
    TRY_CAST(performedDateTime::STRING AS TIMESTAMP),
    TRY_CAST(performedPeriod:start::STRING AS TIMESTAMP)
  ) AS procedure_datetime
  ,CAST(TRY_CAST(performedPeriod:end::STRING AS TIMESTAMP) AS DATE) AS procedure_end_date
  ,TRY_CAST(performedPeriod:end::STRING AS TIMESTAMP) AS procedure_end_datetime
  
  -- Type
  ,32817 AS procedure_type_concept_id
  
  -- Modifier (body site)
  ,0 AS modifier_concept_id
  
  -- Quantity
  ,1 AS quantity
  
  -- References
  ,CASE 
    WHEN performer[0]:actor:reference IS NOT NULL THEN
      ABS(HASH(REGEXP_EXTRACT(performer[0]:actor:reference::STRING, 'Practitioner/(.+)', 1)))
    ELSE NULL
  END AS provider_id
  ,CASE 
    WHEN encounter:reference IS NOT NULL THEN
      ABS(HASH(REGEXP_EXTRACT(encounter:reference::STRING, 'Encounter/(.+)', 1)))
    ELSE NULL
  END AS visit_occurrence_id
  ,NULL AS visit_detail_id
  
  -- Source values
  ,code:coding[0]:code::STRING AS procedure_source_value
  ,0 AS procedure_source_concept_id
  ,bodySite[0]:coding[0]:code::STRING AS modifier_source_value
  
  -- Code system info
  ,code:coding[0]:system::STRING AS procedure_code_system
  ,COALESCE(code:coding[0]:display::STRING, code:text::STRING) AS procedure_display
  
  -- Lineage
  ,procedure_uuid AS fhir_procedure_uuid
  ,bundle_uuid
  
FROM STREAM(" || catalog_use || "." || silver_schema || ".procedure)
WHERE status::STRING = 'completed'
  AND code IS NOT NULL
";

SELECT create_procedure_stmt AS statement;

In [ ]:
EXECUTE IMMEDIATE create_procedure_stmt;

In [ ]:
-- Verify procedure_occurrence table
SELECT 
  procedure_occurrence_id,
  person_id,
  procedure_concept_id,
  procedure_date,
  procedure_source_value,
  procedure_display
FROM procedure_occurrence
LIMIT 10;

In [ ]:
-- Procedure vocabulary distribution
SELECT 
  CASE 
    WHEN procedure_code_system LIKE '%snomed%' THEN 'SNOMED CT'
    WHEN procedure_code_system LIKE '%cpt%' THEN 'CPT-4'
    WHEN procedure_code_system LIKE '%hcpcs%' THEN 'HCPCS'
    WHEN procedure_code_system LIKE '%icd-10-pcs%' THEN 'ICD-10-PCS'
    ELSE procedure_code_system
  END AS vocabulary,
  COUNT(*) AS count
FROM procedure_occurrence
GROUP BY vocabulary
ORDER BY count DESC;

In [ ]:
-- Top procedures
SELECT 
  procedure_source_value,
  procedure_display,
  COUNT(*) AS occurrences
FROM procedure_occurrence
GROUP BY procedure_source_value, procedure_display
ORDER BY occurrences DESC
LIMIT 20;